In [9]:
import pandas as pd
import numpy as np
import os

In [10]:
movies = pd.read_csv('../data/raw/movie.csv')
ratings = pd.read_csv('../data/raw/rating.csv')

In [11]:
# Get unique users
unique_users = ratings['userId'].unique()

# Set random seed for reproducibility and pick 10% of users
np.random.seed(42)
sampled_users = np.random.choice(
    unique_users, 
    size=int(len(unique_users) * 0.10), 
    replace=False
)

# Filter ratings to keep only sampled users
small_ratings = ratings[ratings['userId'].isin(sampled_users)].copy()

# Filter out noise (inactive users and unpopular movies)
USER_MIN_RATINGS = 20
MOVIE_MIN_RATINGS = 5

# Keep users with at least 20 ratings
user_counts = small_ratings['userId'].value_counts()
active_users = user_counts[user_counts >= USER_MIN_RATINGS].index
filtered_ratings = small_ratings[small_ratings['userId'].isin(active_users)]

# Keep movies with at least 5 ratings
movie_counts = filtered_ratings['movieId'].value_counts()
popular_movies = movie_counts[movie_counts >= MOVIE_MIN_RATINGS].index
filtered_ratings = filtered_ratings[filtered_ratings['movieId'].isin(popular_movies)]

print(f"Final shape of ratings: {filtered_ratings.shape}")

Final shape of ratings: (1998669, 4)


In [12]:
# Filter movies to match our cleaned ratings dataset
valid_movie_ids = filtered_ratings['movieId'].unique()
filtered_movies = movies[movies['movieId'].isin(valid_movie_ids)].copy()

# Process genres for LightFM item features
# Split the string 'Action|Adventure|Sci-Fi' into a list: ['Action', 'Adventure', 'Sci-Fi']
filtered_movies['genres'] = filtered_movies['genres'].str.split('|')

# Handle missing genres (MovieLens uses '(no genres listed)')
filtered_movies['genres'] = filtered_movies['genres'].apply(
    lambda x: [] if x == ['(no genres listed)'] else x
)

In [13]:
filtered_movies.head()

,movieId,title,genres
0,1,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]"
1,2,Jumanji (1995),"[Adventure, Children, Fantasy]"
2,3,Grumpier Old Men (1995),"[Comedy, Romance]"
3,4,Waiting to Exhale (1995),"[Comedy, Drama, Romance]"
4,5,Father of the Bride Part II (1995),[Comedy]


In [14]:
output_dir = '../data/preprocessed'

# Save the cleaned ratings
filtered_ratings.to_csv(f'{output_dir}/ratings_cleaned.csv', index=False)

# Save the cleaned movies and their features (genres)
filtered_movies.to_csv(f'{output_dir}/movies_cleaned.csv', index=False)